# Analista Lab · Modelos y validación

600 observaciones sintéticas independientes. No representan clientes reales. Objetivo: practicar clasificación, referencia y selección sin fuga de información. No sirve para recomendar decisiones reales. Dependencias: numpy, pandas, scikit-learn; normalmente disponibles en Colab. En local: `pip install numpy pandas scikit-learn`.

Seleccionaremos C mediante validación cruzada dentro de entrenamiento. La prueba se consulta al final para evaluar el modelo elegido y la referencia.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score, confusion_matrix

X_array, y_array = make_classification(n_samples=600, n_features=6, n_informative=4, n_redundant=1, weights=[0.8, 0.2], random_state=42)
X = pd.DataFrame(X_array, columns=[f"feature_{i}" for i in range(6)])
y = pd.Series(y_array, name="clase")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
assert set(X_train.index).isdisjoint(set(X_test.index))
print("Entrenamiento:", len(X_train), "Prueba reservada:", len(X_test))


## Validación dentro de entrenamiento

Cada fold ajusta imputación y escalado únicamente sobre sus filas de entrenamiento porque están dentro del pipeline. La selección compara balanced accuracy; revisa después si esa métrica coincide con el coste de los errores del problema.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []
for valor_c in [0.1, 1.0, 10.0]:
    candidato = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=valor_c, max_iter=2000, random_state=42))
    scores = cross_val_score(candidato, X_train, y_train, cv=cv, scoring="balanced_accuracy")
    resultados.append({"C": valor_c, "cv_media": float(scores.mean()), "cv_sd": float(scores.std(ddof=1))})
comparacion = pd.DataFrame(resultados).sort_values("cv_media", ascending=False)
mejor_c = float(comparacion.iloc[0]["C"])
print(comparacion.to_string(index=False))
print("C seleccionado sin consultar prueba:", mejor_c)


## Cierra las decisiones antes de evaluar

No cambies C, las variables ni el umbral usando los resultados de la siguiente celda. Si haces nuevas decisiones después, necesitarás otra prueba independiente para una evaluación final honesta.


In [ ]:
modelo = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=mejor_c, max_iter=2000, random_state=42))
referencia = DummyClassifier(strategy="most_frequent")
modelo.fit(X_train, y_train)
referencia.fit(X_train, y_train)

evaluacion = []
for nombre, estimador in [("Referencia", referencia), ("Logística", modelo)]:
    pred = estimador.predict(X_test)
    evaluacion.append({"modelo": nombre, "balanced_accuracy": balanced_accuracy_score(y_test, pred), "precision": precision_score(y_test, pred, zero_division=0), "recall": recall_score(y_test, pred, zero_division=0)})
print(pd.DataFrame(evaluacion).to_string(index=False))
print("Matriz logística (filas=reales, columnas=predichas):")
print(confusion_matrix(y_test, modelo.predict(X_test), labels=[0, 1]))


## Reto · Defiende el informe

Si el modelo no supera la referencia en prueba, no lo presentes como una mejora. Informa el resultado y diseña un nuevo ciclo de evaluación con otra prueba independiente antes de hacer ajustes.

1. Explica por qué el split aleatorio estratificado es razonable para estos datos sintéticos independientes, pero puede fallar con datos temporales o repetidos por cliente.
2. Compara referencia y modelo. Describe qué errores importa minimizar.
3. Propón una mejora y cómo evaluarla sin reutilizar la prueba para seleccionar.
4. Explica por qué un resultado sintético no demuestra desempeño en una empresa real.


In [ ]:
# Añade comprobaciones o análisis exploratorio de entrenamiento.
# No utilices X_test/y_test para elegir nuevas variables o parámetros.


## Extensión · PCA y clustering exploratorio

Estos son ejemplos sobre entrenamiento. PCA no usa y; la varianza explicada no es accuracy. Estandarizamos porque las distancias y la varianza dependen de la escala. No se presentan estos grupos como segmentos de negocio validados.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pca_pipeline = make_pipeline(StandardScaler(), PCA(n_components=0.80, svd_solver="full"))
componentes = pca_pipeline.fit_transform(X_train)
print("Componentes para al menos 80 % de varianza:", componentes.shape[1])
print("Varianza acumulada:", pca_pipeline[-1].explained_variance_ratio_.sum())
cluster_pipeline = make_pipeline(StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=42))
clusters = cluster_pipeline.fit_predict(X_train)
print("Tamaños de grupos exploratorios:", np.bincount(clusters))


Referencia oficial para evitar fuga de información: https://scikit-learn.org/stable/common_pitfalls.html

Referencia de PCA: https://scikit-learn.org/stable/modules/decomposition.html#pca
